In [1]:
import numpy as np
import pandas as pd

from statsforecast import StatsForecast
from statsforecast.models import Naive, HistoricAverage, ARIMA

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_csv("../challenge_data/vm_cpu_readings.csv.gz")   
agg_cpu = df.groupby("timestamp")["avg_cpu"].mean()

train_cpu, hold_cpu = agg_cpu.iloc[:110], agg_cpu.iloc[110:]   
sf_df = pd.DataFrame({"unique_id": "agg_cpu", "ds": np.arange(110), "y": train_cpu.values})

In [3]:
sf = StatsForecast(
    models=[Naive(), HistoricAverage(), ARIMA(order=(3, 0, 0), season_length=1, alias="AR3")],
    freq=1,
)
cv = sf.cross_validation(df=sf_df, h=24, step_size=4, n_windows=10)
print(sorted(cv["cutoff"].unique()))   # expect 49, 53, ..., 85
print(cv["ds"].max())                  # expect 109

[np.int64(49), np.int64(53), np.int64(57), np.int64(61), np.int64(65), np.int64(69), np.int64(73), np.int64(77), np.int64(81), np.int64(85)]
109


In [ ]:
cv["step"] = cv.groupby("cutoff").cumcount() + 1
h = cv["step"]

cv["Blend50"] = 0.5 * cv["Naive"] + 0.5 * cv["HistoricAverage"]

w_lin = 1 - (h - 1) / 23                                  
cv["BlendLin"] = w_lin * cv["Naive"] + (1 - w_lin) * cv["HistoricAverage"]

taus = [2, 4, 8, 12, 24]
for tau in taus:
    w = np.exp(-h / tau)                                   
    cv[f"BlendExp{tau}"] = w * cv["Naive"] + (1 - w) * cv["HistoricAverage"]

In [5]:
models = ["Naive", "HistoricAverage", "AR3", "Blend50", "BlendLin"] + [f"BlendExp{t}" for t in taus]

rows = []
for m in models:
    e = cv["y"] - cv[m]
    rows.append({"model": m, "MAE": e.abs().mean(), "RMSE": (e**2).mean() ** 0.5})
res = pd.DataFrame(rows).set_index("model")
res["RMSE_vs_Naive"] = res["RMSE"] / res.loc["Naive", "RMSE"]
print(res.round(4))

fold = (cv.assign(**{m: (cv["y"] - cv[m])**2 for m in models})
          .groupby("cutoff")[models].mean() ** 0.5)
print("mean fold RMSE:\n", fold.mean().round(4))
print("worst fold RMSE:\n", fold.max().round(3))

                    MAE    RMSE  RMSE_vs_Naive
model                                         
Naive            0.2392  0.2920         1.0000
HistoricAverage  0.2037  0.2400         0.8221
AR3              0.1918  0.2306         0.7899
Blend50          0.2028  0.2382         0.8158
BlendLin         0.1853  0.2201         0.7537
BlendExp2        0.1940  0.2297         0.7868
BlendExp4        0.1862  0.2209         0.7567
BlendExp8        0.1826  0.2154         0.7377
BlendExp12       0.1853  0.2180         0.7466
BlendExp24       0.1983  0.2337         0.8005
mean fold RMSE:
 Naive              0.2733
HistoricAverage    0.2329
AR3                0.2220
Blend50            0.2337
BlendLin           0.2140
BlendExp2          0.2225
BlendExp4          0.2141
BlendExp8          0.2099
BlendExp12         0.2129
BlendExp24         0.2273
dtype: float64
worst fold RMSE:
 Naive              0.496
HistoricAverage    0.337
AR3                0.337
Blend50            0.320
BlendLin           0.321
B

In [6]:
cut = sorted(cv["cutoff"].unique())
early, late = cut[:5], cut[5:]

def rmse(d, m):
    return ((d["y"] - d[m]) ** 2).mean() ** 0.5

for a, b, name in [(early, late, "choose on early, test on late"),
                   (late, early, "choose on late, test on early")]:
    da, db = cv[cv["cutoff"].isin(a)], cv[cv["cutoff"].isin(b)]
    best = min([f"BlendExp{t}" for t in taus], key=lambda m: rmse(da, m))
    print(name, "->", best,
          "| test RMSE:", round(rmse(db, best), 4),
          "| HistoricAverage:", round(rmse(db, "HistoricAverage"), 4),
          "| AR3:", round(rmse(db, "AR3"), 4))

choose on early, test on late -> BlendExp8 | test RMSE: 0.2021 | HistoricAverage: 0.2005 | AR3: 0.2003
choose on late, test on early -> BlendExp2 | test RMSE: 0.2564 | HistoricAverage: 0.2739 | AR3: 0.2574


Tuned blends looked best pooled (RMSE 0.215), but a split-half test showed the tuned parameter was unstable and its advantage disappeared out of sample. We therefore select [BlendLin / AR(3)], which has no parameter fitted to the validation folds. Differences among the top candidates are small relative to fold-to-fold variation.

In [7]:
cands = ["Naive", "HistoricAverage", "AR3", "Blend50", "BlendLin"]
for name, folds in [("early (49-65)", early), ("late (69-85)", late)]:
    d = cv[cv["cutoff"].isin(folds)]
    print(name, {m: round(rmse(d, m), 4) for m in cands})

early (49-65) {'Naive': np.float64(0.3441), 'HistoricAverage': np.float64(0.2739), 'AR3': np.float64(0.2574), 'Blend50': np.float64(0.2649), 'BlendLin': np.float64(0.2315)}
late (69-85) {'Naive': np.float64(0.2282), 'HistoricAverage': np.float64(0.2005), 'AR3': np.float64(0.2003), 'Blend50': np.float64(0.2081), 'BlendLin': np.float64(0.208)}
